# Search Baselines
So sánh keyword overlap, TF-IDF và hybrid ranking trên một corpus nhỏ.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

papers = pd.DataFrame([
 {'title':'Retrieval-Augmented Generation','abstract':'retrieval and generation with external evidence','citations':18000,'year':2020},
 {'title':'Sentence-BERT','abstract':'sentence embeddings for semantic similarity','citations':20000,'year':2019},
 {'title':'Marine Biology','abstract':'fish populations and ocean ecosystems','citations':500,'year':2022},
])
query='retrieval augmented generation evidence'
texts=(papers.title+' '+papers.abstract).tolist()

In [ ]:
vectorizer=TfidfVectorizer(ngram_range=(1,2), stop_words='english')
matrix=vectorizer.fit_transform([query,*texts])
tfidf=cosine_similarity(matrix[0],matrix[1:]).ravel()
keyword=np.array([len(set(query.split()) & set(text.lower().split()))/len(set(query.split())) for text in texts])
citation=np.log1p(papers.citations)/np.log1p(papers.citations.max())
recency=np.exp(-(2026-papers.year)/8)
papers.assign(keyword=keyword, tfidf=tfidf, hybrid=.35*keyword+.40*tfidf+.15*citation+.10*recency).sort_values('hybrid',ascending=False)

In [ ]:
def dcg(relevance):
    return sum((2**rel-1)/np.log2(i+2) for i,rel in enumerate(relevance))

labels={0:3,1:1,2:0}
ranking=np.argsort(-(.35*keyword+.40*tfidf+.15*citation+.10*recency))
ndcg=dcg([labels[i] for i in ranking])/dcg(sorted(labels.values(), reverse=True))
ndcg

## Kết luận
Hybrid ranking cân bằng truy vấn, độ tương đồng và ảnh hưởng. Trên dữ liệu thật cần query set do người đánh giá gán relevance.